## Baseline-Modell

# Erstellt je User einen Case/Trace mit den dazugehörigen Events (Zustandsverläufen).
# Insgesamt 3 Zustände möglich.
Ob die Patienten Tinnitus hatten oder nicht, wird noch nicht beachtet.

In [ ]:
import pandas as pd

In [ ]:
# CSV laden
df = pd.read_csv('/Users/elias/Desktop/Uni/4. Semester : Masterarbeit/2026-04-02 TYT_answers2.csv', sep=";")

# Überblick verschaffen

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

In [ ]:
# Einzigartige Fälle
print("Anzahl Cases:", df["id"].nunique())

In [ ]:
# Einzigartige User
print("Anzahl User:", df["user_id"].nunique())

In [ ]:
# Einträge je User
print(f"Durchschnittliche Einträge pro User: {112184/3339:.2f}")

In [ ]:
# Fehlende Werte finden
df.isna().sum()

# Aufräumen

In [ ]:
# Schauen, ob zwei Spalten redundante Inhalte haben
# bei True sind die Inhalte 1:1 gleich
df["created_at"].equals(df["updated_at"])

In [ ]:
# Schauen, ob die Infos in der Spalte überhaupt relevant sind
df["user_agent"]

In [ ]:
# Schauen, ob die Infos in der Spalte überhaupt relevant sind
df["autosaved"]

In [ ]:
# Spalten "user_agent", "updated_at" und "autosaved" entfernen.
# zudem "notification_fixed" und "notification_date" entfernen, da Miriam sie für unwichtig erklärt hat
df = df.drop(columns=["user_agent", "updated_at", "autosaved", "notification_fixed", "notification_date"])

In [ ]:
df.columns

In [ ]:
# Schauen, dass Werte von Q1-Q8 im erlaubten Bereich sind
# q1 und q8 (0 oder 1)
invalid_q1_q8 = df[
    (df["q1"] < 0) | (df["q1"] > 1) |
    (df["q8"] < 0) | (df["q8"] > 1)
]
print("q1/q8 Fehler:", len(invalid_q1_q8))

# q2 bis q7 (0–10)
invalid_q2_q7 = df[
    (df[["q2","q3","q4","q5","q6","q7"]] < 0).any(axis=1) |
    (df[["q2","q3","q4","q5","q6","q7"]] > 10).any(axis=1)
]
print("q2–q7 Fehler:", len(invalid_q2_q7))

In [ ]:
# Schauen, dass "age" nicht <0 oder >110 ist
df["age"] = pd.to_numeric(df["age"], errors="coerce")
invalid_age = df[(df["age"] < 0) | (df["age"] > 110)]

print(invalid_age[["id", "user_id", "age"]])

In [ ]:
# Sicherstellen, dass "age" numerisch ist
df["age"] = pd.to_numeric(df["age"], errors="coerce")

# Alter und Onset_duration auf NaN setzen
df.loc[(df["user_id"] == 835) & ((df["age"] < 0) | (df["age"] > 110)), "age"] = pd.NA
df.loc[(df["user_id"] == 835), "onset_duration"] = pd.NA

# Kontrolle
df[df["user_id"] == 835]

# Daten aufbereiten

In [ ]:
# Zeitspalten konvertieren
time_cols = ["save_date", "created_at"]

for col in time_cols:
    df[col] = pd.to_datetime(df[col], format="%d.%m.%y %H:%M", errors="coerce")

# Geburtsdatum
date_cols = ["date_of_birth2", "onset_date2"]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], format="%d.%m.%y", errors="coerce")

In [ ]:
df[df["created_at"].isna()]

In [ ]:
df[df["save_date"].isna()]

In [ ]:
# Spalte "save_date" entfernen. Wir werden "created_at" als Zeitstempel nehmen
df = df.drop(columns=["save_date"])

In [ ]:
# Änderungen kontrollieren
df.head()

In [ ]:
import matplotlib.pyplot as plt

# pro User genau ein Eintrag (z. B. erster Eintrag)
df_user = df.drop_duplicates(subset="user_id")

plt.figure(figsize=(20,5))

df_user["age"].value_counts().sort_index().plot(kind="bar")

plt.title("Age Distribution (per User)")
plt.xlabel("Age")
plt.ylabel("Number of Users")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Vorbereitung des Dataframes auf die PM4Py Analyse

In [ ]:
# Score berechnen (ohne q1 und q8)
df["score"] = df[["q2","q3","q4","q5","q6","q7"]].sum(axis=1)

# Kategorien definieren
def categorize(score):
    if score <= 20:
        return "low"
    elif score <= 40:
        return "medium"
    else:
        return "high"

df["concept:name"] = df["score"].apply(categorize)

In [ ]:
# Case = Patient
df_event = df.rename(columns={
    "user_id": "case:concept:name",
    "created_at": "time:timestamp"
})

# Datentypen fixen
df_event["case:concept:name"] = df_event["case:concept:name"].astype(str)
df_event["concept:name"] = df_event["concept:name"].astype(str)

# Datentypen Kontrollieren
print(df_event["case:concept:name"].dtype)
print(df_event["concept:name"].dtype)

# Sortieren nach Zeit
df_event = df_event.sort_values(
    by=["case:concept:name", "time:timestamp"]
)

# Relevante Spalten behalten
df_event = df_event[[
    "case:concept:name",
    "concept:name",
    "time:timestamp",
    "age",
    "gender",
    "sound_env"
]]

df_event

In [54]:
# Personen je Level
df_event.groupby("concept:name")["case:concept:name"].nunique()

concept:name
high       466
low       1766
medium    2932
Name: case:concept:name, dtype: int64

In [55]:
# Traces je Level
df_event.groupby("concept:name")["concept:name"].count()

concept:name
high       3584
low       34995
medium    73611
Name: concept:name, dtype: int64

In [53]:
(df_event["case:concept:name"].value_counts() == 1).sum()

np.int64(1018)

In [ ]:
# oder das
# hier viel weniger Attribute mitgenommen
'''
question_cols = ["q1","q2","q3","q4","q5","q6","q7","q8"]

df_event = df.melt(
id_vars=["id", "created_at"],
value_vars=question_cols,
var_name="concept:name",
value_name="value"
)

# Spalten für PM4Py umbenennen
df_event = df_event.rename(columns={
"id": "case:concept:name",
"created_at": "time:timestamp"
})

df_event.head()
'''

In [ ]:
''' # Das ist eigentlich redundant. Aber erstmal behalten
# Beide Spalten auf string casten
df_agg = df.groupby("id").agg({ "age": "first" }).reset_index()

df_agg["id"] = df_agg["id"].astype(str)
df_event["case:concept:name"] = df_event["case:concept:name"].astype(str)

# Merge durchführen
df_event = df_event.merge(
    df_agg,
    left_on="case:concept:name",
    right_on="id",
    how="left"
)

# Doppelte Spalten nach Merge entfernen
df_event = df_event.drop(columns=["id", "age_y"])

# age_x zu age umbenennen
df_event = df_event.rename(columns={"age_x": "age"})

df_event
'''

In [ ]:
df_event[df_event["time:timestamp"].isna()]

In [ ]:
# Kontrollieren, dass jetzt die Sortierung nach Case und timestamp sortiert ist
df_event.head(20)

In [ ]:
import pm4py

log = pm4py.convert_to_event_log(df_event)

print("Anzahl Traces:", len(log))

In [ ]:
# euren gewünschten Pfad hernehmen. Dann auch hernehmen, um damit weiter zu arbeiten

pm4py.write_xes(log, '/Users/elias/Desktop/Uni/4. Semester : Masterarbeit/2026-04-06 TYT_event_log V2.xes')